[Теория 1](https://colab.research.google.com/drive/1hPiHZOFr4v4LaiCzhecue96RKVSFhhEj?usp=sharing)

[Теория 2](https://colab.research.google.com/drive/1xN62w0kG0J_OqErqFFuh6sWpYN-SFJN3?usp=sharing)

# Задание №1

**Анализ данных о покупках**

- Шаг 1. Импортировать необходимые библиотеки.
- Шаг 2. Получить данные из файла
https://github.com/OlesiaAngel/DataAnalitics/raw/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/dataset6_pr.csv
- Шаг 3. Выполнить кластерный анализ с помощью алгоритма k-means.
- Шаг 4. Постройть диаграмму рассеивания - scatter plot.
- Шаг 5. Сформулировать выводы.


Датасет - информация о покупках клиентов:

- **Товар** - уникальный номер товара
- **Количество** - количество купленных товаров
- **Сумма** - объем продаж этого товара
- **Процент_продаж** - доля продаж этого товара от общего объема продаж
- **Процент_колво** -  доля продаж этого товара от общего количества проданных товаров
- **Сумма_продаж**- комулятивная сумма по объему продаж
- **Сумма_колво** - комулятивная сумма по количеству проданных товаров


In [ ]:
#Ваш код здесь

# Шаг 1. Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Шаг 2. Загрузка данных
url = 'https://github.com/OlesiaAngel/DataAnalitics/raw/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/dataset6_pr.csv'
df = pd.read_csv(url, sep=';')

print('Данные о покупках:')
display(df.head(10))
print(f'\nФорма датасета: {df.shape}')
print(f'\nТипы данных:\n{df.dtypes}')
print(f'\nОписание:\n')
display(df.describe())


In [ ]:
# Шаг 3. K-means кластерный анализ
# Выбираем признаки для кластеризации
feature_cols = [c for c in df.columns if df[c].dtype in ['float64', 'int64']]
feature_cols = [c for c in feature_cols if c != 'Товар']
X = df[feature_cols].dropna()

print(f'Признаки для кластеризации: {feature_cols}')
print(f'Данных для кластеризации: {X.shape}')

# Нормализация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Метод локтя для выбора оптимального k
inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_range, inertias, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Инерция (SSE)')
plt.title('Метод локтя для выбора k')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Применяем K-means с оптимальным k=3 (ABC-анализ: A, B, C группы товаров)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_cluster = df.copy()
df_cluster = df_cluster.dropna(subset=feature_cols)
df_cluster['Кластер'] = kmeans.fit_predict(X_scaled)

# Маппинг кластеров по сумме продаж (A=высокие, B=средние, C=низкие)
cluster_means = df_cluster.groupby('Кластер')['Процент_продаж'].mean().sort_values(ascending=False)
cluster_map = {cluster_means.index[0]: 'A (высокие продажи)',
               cluster_means.index[1]: 'B (средние продажи)',
               cluster_means.index[2]: 'C (низкие продажи)'}
df_cluster['Группа'] = df_cluster['Кластер'].map(cluster_map)

print('Результаты кластеризации:')
display(df_cluster.groupby('Группа')[feature_cols].mean().round(2))
print('\nКоличество товаров в каждой группе:')
print(df_cluster['Группа'].value_counts())


In [ ]:
# Шаг 4. Диаграмма рассеивания (scatter plot)
plt.figure(figsize=(12, 7))

colors = {'A (высокие продажи)': '#e74c3c',
          'B (средние продажи)': '#f39c12',
          'C (низкие продажи)': '#2ecc71'}

x_col = feature_cols[0] if len(feature_cols) > 0 else 'Процент_продаж'
y_col = feature_cols[1] if len(feature_cols) > 1 else 'Процент_колво'

for group, color in colors.items():
    subset = df_cluster[df_cluster['Группа'] == group]
    plt.scatter(subset[x_col], subset[y_col],
                label=group, color=color, s=80, alpha=0.8, edgecolors='white', linewidth=0.5)

plt.xlabel(x_col, fontsize=12)
plt.ylabel(y_col, fontsize=12)
plt.title('K-means кластеризация товаров (k=3)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Вывод

В результате кластерного анализа методом K-means все товары были разделены на **3 группы (ABC-анализ)**:

- **Группа A (высокие продажи)** — товары с наибольшей долей в объёме продаж. Требуют приоритетного внимания и постоянного контроля запасов.
- **Группа B (средние продажи)** — товары со средними показателями. Нуждаются в периодическом мониторинге.
- **Группа C (низкие продажи)** — товары с наименьшим вкладом в общий оборот. Можно рассмотреть снижение запасов или отказ от них.

Алгоритм k-means эффективно разделил товары на группы по параметрам продаж и количества, что позволяет оптимизировать управление ассортиментом.



## Задача № 2

Прежде чем писать код, необходимо разобраться в решаемой задаче и доступных данных. В этом проекте мы будем работать с  данными об энергоэффективности зданий в Нью-Йорке.

Наша цель: подготовить данные для построения модели прогнозирования баллов Energy Star Score.
Выполним следующие этапы:
*  Очистка и форматирование данных.
*  Разведочный анализ данных.
*  Конструирование и выбор признаков.
*  Выбор метрики для моделей машинного обучения.

## Импортируем необходимые библиотеки (Imports)

Мы будем использовать стандартные библиотеки data science и машинного обучения: `numpy`, `pandas`,  `scikit-learn`. А также библиотеки `matplotlib` and `seaborn` для визуализации.

In [ ]:
# Pandas and numpy for data manipulation
import pandas as pd
import numpy as np

# No warnings about setting value on copy of slice
pd.options.mode.chained_assignment = None

# Display up to 60 columns of a dataframe
pd.set_option('display.max_columns', 60)

# Matplotlib visualization
import matplotlib.pyplot as plt
%matplotlib inline

# Set default font size
plt.rcParams['font.size'] = 24

# Internal ipython tool for setting figure size
from IPython.core.pylabtools import figsize

# Seaborn for visualization
import seaborn as sns
sns.set(font_scale = 2)

# Splitting data into training and testing
from sklearn.model_selection import train_test_split


# Очистка данных

Далеко не каждый набор данных представляет собой идеально подобранное множество наблюдений, без аномалий и пропущенных значений. В реальных данных мало порядка, так что прежде чем приступить к анализу, их нужно очистить и привести к приемлемому формату. Очистка данных — неприятная, но обязательная процедура при решении большинства задач по анализу данных.

Сначала можно загрузить данные в виде кадра данных (dataframe) Pandas и изучить их:

In [ ]:
# Read in data into a dataframe
data = pd.read_csv('https://github.com/OlesiaAngel/DataAnalitics/blob/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/examples/Energy.csv?raw=true')

# Display top of dataframe
data.head()


### Типы данных и обработка пропусков

Есть еще одна проблема — отсутствующие значения, помеченные как «Not Available». Это строковое значение в Python, которое означает, что даже строки с числами будут храниться как типы данных object, потому что если в колонке есть какая-нибудь строковая, Pandas конвертирует её в колонку, полностью состоящую из строковых. Типы данных колонок можно узнать с помощью метода dataframe.info():

In [ ]:
# ваш код здесь
print('Размер датасета:', data.shape)
print('\nИнформация о столбцах:')
data.info()


Наверняка некоторые колонки, которые явно содержат числа, сохранены как объекты. Мы не можем применять числовой анализ к строковым значениям, так что конвертируем их в числовые типы данных (особенно float)!

### Преобразование данных к нужному типу

Необходимо сначала заменить все «Not Available» на not a number (np.nan), которые можно интерпретировать как числа, а затем конвертирует содержимое определённых колонок в тип float:

In [ ]:
# ваш код здесь

# Заменяем строковые 'Not Available' на np.nan
data = data.replace('Not Available', np.nan)

# Конвертируем числовые столбцы в float
# Определяем столбцы, которые должны быть числовыми
for col in data.columns:
    try:
        data[col] = pd.to_numeric(data[col], errors='ignore')
    except Exception:
        pass

print('Типы данных после преобразования:')
print(data.dtypes)


Выведем статистические данные:

In [ ]:
# ваш код здесь
print('Статистическое описание числовых столбцов:')
display(data.describe())


Когда значения в соответствующих колонках у нас станут числами, можно начинать исследовать данные.

### Отсутствующие и аномальные данные

Наряду с некорректными типами данных одна из самых частых проблем — отсутствующие значения. Они могут отсутствовать по разным причинам, и перед обучением модели эти значения нужно либо заполнить, либо удалить. Сначала давайте выясним, сколько у нас не хватает значений в каждой колонке

In [ ]:
# ваш код здесь

# Процент пропущенных значений по каждому столбцу
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_df = pd.DataFrame({'Пропущено': missing, 'Процент (%)': missing_pct})
missing_df = missing_df[missing_df['Пропущено'] > 0].sort_values('Процент (%)', ascending=False)

print(f'Всего столбцов: {data.shape[1]}')
print(f'Столбцов с пропущенными значениями: {len(missing_df)}')
display(missing_df.head(20))


Убирать информацию всегда нужно с осторожностью, и если много значений в колонке отсутствует, то она, вероятно, не пойдёт на пользу нашей модели. Порог, после которого колонки лучше выкидывать, зависит от вашей задачи, а в нашем проекте мы будем удалять колонки, пустые более чем на половину.

In [ ]:
# ваш код здесь

# Удаляем столбцы, в которых более 50% пропущенных значений
threshold = 0.5
cols_before = data.shape[1]
data = data.dropna(axis=1, thresh=int((1 - threshold) * len(data)))
cols_after = data.shape[1]

print(f'Столбцов до удаления: {cols_before}')
print(f'Столбцов после удаления (порог >50% пропусков): {cols_after}')
print(f'Удалено столбцов: {cols_before - cols_after}')
print(f'\nОставшиеся столбцы: {data.columns.tolist()}')


# Разведочный анализ данных

Скучный, но необходимый этап очистки данных закончен, можно перейти к исследованию! Разведочный анализ данных (РАД) — неограниченный по времени процесс, в ходе которого мы вычисляем статистику и ищем в данных тенденции, аномалии, шаблоны или взаимосвязи.

Коротко говоря, РАД — это попытка выяснить, что нам могут сказать данные. Обычно анализ начинается с поверхностного обзора, затем мы находим интересные фрагменты и анализируем их подробнее. Выводы могут быть интересными сами по себе, или они могут способствовать выбору модели, помогая решить, какие признаки мы будем использовать.

### Однопеременные графики

Наша цель — прогнозировать значение Energy Star Score (в наших данных переименовано в score), так что имеет смысл начать с исследования распределения этой переменной. Гистограмма — простой, но эффективный способ визуализации распределения одиночной переменной, и её можно легко построить с помощью matplotlib.

In [ ]:
# ваш код здесь

# Определяем столбец с Energy Star Score
score_col = None
for col in data.columns:
    if 'score' in col.lower() or 'energy star' in col.lower():
        score_col = col
        break

if score_col is None:
    score_col = [c for c in data.columns if 'Score' in c or 'score' in c]
    score_col = score_col[0] if score_col else data.select_dtypes(include=[np.number]).columns[0]

# Переименуем для удобства
data = data.rename(columns={score_col: 'score'})
score_col = 'score'

print(f'Целевая переменная: score (исходное название: {score_col})')
print(f'Описание score:\n{data["score"].describe()}')

plt.style.use('fivethirtyeight')
figsize(8, 8)

data['score'].dropna().hist(bins=100, edgecolor='k', color='steelblue')
plt.xlabel('Оценка (score)')
plt.ylabel('Количество зданий')
plt.title('Распределение Energy Star Score')
plt.show()


### Поиск взаимосвязей

Главная часть РАД — поиск взаимосвязей между признаками и нашей целью. Коррелирующие с ней переменные полезны для использования в модели, потому что их можно применять для прогнозирования. Один из способов изучения влияния категориальной переменной (которая принимает только ограниченный набор значений) на цель — это построить график плотности с помощью библиотеки Seaborn.

График плотности можно считать сглаженной гистограммой, потому что он показывает распределение одиночной переменной. Можно раскрасить отдельные классы на графике, чтобы посмотреть, как категориальная переменная меняет распределение. Этот код строит график плотности Energy Star Score, раскрашенный в зависимости от типа здания (для списка зданий с более чем 100 измерениями):

In [ ]:
# ваш код здесь

# Определяем столбец типа здания
type_col = None
for col in data.columns:
    if 'type' in col.lower() and 'property' in col.lower():
        type_col = col
        break
if type_col is None:
    type_col = [c for c in data.columns if 'type' in c.lower() or 'use' in c.lower()]
    type_col = type_col[0] if type_col else None

if type_col:
    # Типы зданий с более чем 100 наблюдениями
    types = data[type_col].value_counts()
    types = types[types > 100].index.tolist()
    plot_data = data[data[type_col].isin(types)].dropna(subset=['score', type_col])

    figsize(12, 10)
    for btype in types[:8]:  # до 8 типов
        subset = plot_data[plot_data[type_col] == btype]['score']
        subset.plot.kde(label=btype[:30], linewidth=2)

    plt.xlabel('score')
    plt.ylabel('Плотность')
    plt.title('Energy Star Score по типу здания')
    plt.legend(loc='upper left', fontsize=10)
    plt.xlim(0, 100)
    plt.show()
else:
    print('Столбец типа здания не найден. Доступные столбцы:', data.columns.tolist())


Аналогичный график можно использовать для оценки Energy Star Score по районам города:

In [ ]:
# ваш код здесь

# Определяем столбец района
borough_col = None
for col in data.columns:
    if 'borough' in col.lower() or 'neighborhood' in col.lower() or 'district' in col.lower():
        borough_col = col
        break

if borough_col:
    boroughs = data[borough_col].value_counts()
    boroughs = boroughs[boroughs > 50].index.tolist()
    plot_data_b = data[data[borough_col].isin(boroughs)].dropna(subset=['score', borough_col])

    figsize(12, 10)
    for borough in boroughs:
        subset = plot_data_b[plot_data_b[borough_col] == borough]['score']
        subset.plot.kde(label=str(borough)[:30], linewidth=2)

    plt.xlabel('score')
    plt.ylabel('Плотность')
    plt.title('Energy Star Score по районам города')
    plt.legend(loc='upper left', fontsize=12)
    plt.xlim(0, 100)
    plt.show()
else:
    print('Столбец района не найден. Доступные столбцы:', data.columns.tolist())


### Корреляции между объектами и целью

Чтобы посчитать взаимосвязи между переменными, можно использовать коэффициент корреляции Пирсона. Это мера интенсивности и направления линейной зависимости между двумя переменными. Значение +1 означает идеально линейную положительную зависимость, а -1 означает идеально линейную отрицательную зависимость.
Хотя этот коэффициент не может отражать нелинейные зависимости, с него можно начать оценку взаимосвязей переменных. В Pandas можно легко вычислить корреляции между любыми колонками в кадре данных (dataframe):

In [ ]:
# ваш код здесь

# Корреляции числовых признаков с целевой переменной score
correlations = data.select_dtypes(include=[np.number]).corr()['score'].dropna()
correlations = correlations.drop('score').sort_values()

print('Корреляции признаков со значением score:')
print(correlations)

# Визуализация корреляций
figsize(8, 10)
correlations.plot.barh(color=['#e74c3c' if x < 0 else '#2ecc71' for x in correlations.values])
plt.title('Корреляция признаков с Energy Star Score')
plt.xlabel('Коэффициент корреляции Пирсона')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


### Двухпеременные графики

Воспользуемся диаграммами рассеивания для визуализации взаимосвязей между двумя непрерывными переменными. К цветам точек можно добавить дополнительную информацию, например, категориальную переменную. Ниже показана взаимосвязь Energy Star Score и EUI, цветом обозначены разные типы зданий:

In [ ]:
# ваш код здесь

# Определяем EUI столбец
eui_col = None
for col in data.columns:
    if 'site eui' in col.lower() or ('eui' in col.lower() and 'source' not in col.lower()):
        eui_col = col
        break

numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if 'score' in numeric_cols:
    numeric_cols.remove('score')

plot_col = eui_col if eui_col else (numeric_cols[0] if numeric_cols else None)

if plot_col:
    figsize(10, 8)
    plot_data_s = data[[plot_col, 'score']].dropna()
    # Ограничиваем выбросы
    q99_x = plot_data_s[plot_col].quantile(0.99)
    plot_data_s = plot_data_s[plot_data_s[plot_col] <= q99_x]

    plt.scatter(plot_data_s[plot_col], plot_data_s['score'],
                alpha=0.3, s=20, color='steelblue')
    plt.xlabel(plot_col)
    plt.ylabel('Energy Star Score')
    plt.title(f'Взаимосвязь {plot_col} и Score')
    plt.show()
    print(f'Корреляция {plot_col} со score: {data[[plot_col, "score"]].corr().iloc[0, 1]:.3f}')
else:
    print('Числовые столбцы не найдены')


### Pairs Plot (парный график)

Наш последний исследовательский график называется Pairs Plot (парный график). Это прекрасный инструмент, позволяющий увидеть взаимосвязи между различными парами переменных и распределение одиночных переменных. Мы воспользуемся библиотекой Seaborn и функцией PairGrid для создания парного графика с диаграммой рассеивания в верхнем треугольнике, с гистограммой по диагонали, двухмерной диаграммой плотности ядра и коэффициентов корреляции в нижнем треугольнике.

In [ ]:
# ваш код здесь

# Выбираем до 4 числовых признаков с наибольшей корреляцией + score
top_corr_features = correlations.abs().sort_values(ascending=False).head(4).index.tolist()
cols_for_plot = top_corr_features + ['score']
cols_for_plot = [c for c in cols_for_plot if c in data.columns]

plot_data_p = data[cols_for_plot].dropna()

# Ограничиваем выбросы по квантилям
for col in plot_data_p.columns:
    q01 = plot_data_p[col].quantile(0.01)
    q99 = plot_data_p[col].quantile(0.99)
    plot_data_p = plot_data_p[(plot_data_p[col] >= q01) & (plot_data_p[col] <= q99)]

print(f'Признаки для парного графика: {cols_for_plot}')
print(f'Наблюдений: {len(plot_data_p)}')

figsize(14, 14)
g = sns.PairGrid(plot_data_p)
g.map_upper(plt.scatter, alpha=0.3, s=10)
g.map_diag(plt.hist, bins=30)
g.map_lower(sns.kdeplot)
plt.suptitle('Парный график ключевых признаков', y=1.01)
plt.tight_layout()
plt.show()


# Конструирование признаков (Feature Engineering and Selection)

Конструирование и выбор признаков зачастую приносит наибольшую отдачу с точки зрения времени, потраченного на машинное обучение.

Мы сделаем следующее:

* Применим к категориальным переменным (квартал и тип собственности) one-hot кодирование.
* Добавим взятие натурального логарифма от всех числовых переменных.

One-hot кодирование необходимо для того, чтобы включить в модель категориальные переменные. Алгоритм машинного обучения не сможет понять тип «офис», так что если здание офисное, мы присвоим ему признак 1, а если не офисное, то 0.

Добавление преобразованных признаков поможет модели узнать о нелинейных взаимосвязях внутри данных. В анализе данных является нормальной практикой извлекать квадратные корни, брать натуральные логарифмы или ещё как-то преобразовывать признаки, это зависит от конкретной задачи или вашего знания лучших методик. В данном случае мы добавим натуральный логарифм всех числовых признаков.

Выполним следующие шаги:
* выбирем числовые признаки и вычислим их логарифмы,  
* выбирем два категориальных признака, применяет к ним one-hot кодирование,
* объединим оба множества в одно.
Судя по описанию, предстоит куча работы, но в Pandas всё получается довольно просто!

In [ ]:
# ваш код здесь

# Создаём копию данных для feature engineering
features = data.copy()

# Выбираем строки, где score не пустой
features = features[features['score'].notna()]

# Числовые признаки
numeric_features = features.select_dtypes(include=[np.number]).drop(columns=['score'])

# Логарифм числовых признаков (добавляем 1 чтобы избежать log(0))
log_features = np.log1p(numeric_features.abs())
log_features.columns = ['log_' + col for col in log_features.columns]

# Категориальные признаки — one-hot кодирование
cat_cols = features.select_dtypes(include=['object']).columns.tolist()
print(f'Категориальные столбцы: {cat_cols}')

# Выбираем до 2 категориальных признаков
cat_to_encode = cat_cols[:2] if len(cat_cols) >= 2 else cat_cols
if cat_to_encode:
    cat_encoded = pd.get_dummies(features[cat_to_encode], drop_first=True, dtype=float)
    print(f'One-hot признаков создано: {cat_encoded.shape[1]}')
else:
    cat_encoded = pd.DataFrame()

# Объединяем числовые, логарифмы и категориальные
features_final = pd.concat([numeric_features, log_features, cat_encoded], axis=1)
labels = features['score'].values

print(f'\nИтоговая форма матрицы признаков: {features_final.shape}')
print(f'Количество меток (score): {len(labels)}')


## Выбор признаков

Признаки, которые сильно коррелируют друг с другом, называются коллинеарными. Удаление одной переменной в таких парах признаков часто помогает модели обобщать и быть более интерпретируемой. Обратите внимание, что речь идёт о корреляции одних признаков с другими, а не о корреляции с целью, что только помогло бы нашей модели!

In [ ]:
# ваш код здесь

def remove_collinear_features(df, threshold=0.95):
    """
    Удаляет коллинеарные признаки (|корреляция| > threshold).
    Оставляет первый из пары коррелирующих признаков.
    """
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    df_reduced = df.drop(columns=to_drop)
    print(f'Удалено коллинеарных признаков: {len(to_drop)}')
    print(f'Форма матрицы до: {df.shape} -> после: {df_reduced.shape}')
    return df_reduced

# Удаляем NaN перед анализом корреляций
features_no_na = features_final.fillna(features_final.median(numeric_only=True))
features_reduced = remove_collinear_features(features_no_na, threshold=0.95)
print(f'\nИтоговое количество признаков: {features_reduced.shape[1]}')


# Разбиение на обучающий и тестовый наборы

Прежде чем вычислять базовый уровень, нужно разбить данные на обучающий и тестовый наборы:

1. Обучающий набор признаков — то, что мы предоставляем нашей модели вместе с ответами в ходе обучения. Модель должна выучить соответствие признаков цели.
2. Тестовый набор признаков используется для оценки обученной модели. Когда она обрабатывает тестовый набор, то не видит правильных ответов и должна прогнозировать, опираясь только на доступные признаки. Мы знаем ответы для тестовых данных и можем сравнить с ними результаты прогнозирования.

Для обучения используем 70 % данных, а для тестирования — 30 %:

In [ ]:
# ваш код здесь

# Разбиение 70% обучение / 30% тест
X = features_reduced.copy()
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Обучающий набор: X_train={X_train.shape}, y_train={y_train.shape}')
print(f'Тестовый набор:  X_test={X_test.shape}, y_test={y_test.shape}')


### Выбор метрики. Средняя абсолютная ошибка (mae)

В качестве метрики возьмём среднюю абсолютную ошибку (mae) в прогнозах. Для регрессий есть много других метрик, но хороший совет выбирать какую-то одну метрику и с её помощью оценивать модели. А среднюю абсолютную ошибку легко вычислить и интерпретировать.

In [ ]:
# Function to calculate mean absolute error
# ваш код здесь

def mean_absolute_error(y_true, y_pred):
    """
    Вычисляет среднюю абсолютную ошибку (MAE).
    
    MAE = (1/n) * sum(|y_true - y_pred|)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))

# Базовый уровень: предсказываем медианное значение score
baseline_pred = np.full(len(y_test), fill_value=np.nanmedian(y_train))
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f'Базовый MAE (медиана): {baseline_mae:.2f}')
print(f'Медиана score на обучении: {np.nanmedian(y_train):.2f}')
print(f'Интерпретация: в среднем ошибка составляет {baseline_mae:.1f} баллов Energy Star Score')


Сохраним полученные наборы данных

In [ ]:
# Save the no scores, training, and testing data
# ваш код здесь

import os

# Сохраняем данные с признаками в CSV
X_train.to_csv('train_features.csv', index=False)
X_test.to_csv('test_features.csv', index=False)
pd.Series(y_train, name='score').to_csv('train_labels.csv', index=False)
pd.Series(y_test, name='score').to_csv('test_labels.csv', index=False)

print('Данные сохранены:')
print(f'  train_features.csv — {X_train.shape}')
print(f'  test_features.csv  — {X_test.shape}')
print(f'  train_labels.csv   — {len(y_train)} меток')
print(f'  test_labels.csv    — {len(y_test)} меток')
